# Week 10 extension — extend conditioning + train experiments (notebook 10d)

In Week 10 the conditional diffusion model barely tied the classical
baseline on hard NLL. The 4-D conditioning we fed it
(`area_smoothed`, `mu_universal`, `model_sigma`, `amplitude`) is the *same*
information the classical parametric model already used to produce
`hist_par` — so the diffusion was being told what the subtraction already
absorbed. The goal of this notebook (and its evaluation sibling 10e) is
to find out how much performance reverse diffusion can give us when we
**(a)** feed it conditioning the parametric model didn't already absorb
and **(b)** inject it through a stronger mechanism than raw concat.

This is a *staged* ablation study. You will progress through the
experiment menu one knob at a time — never enabling two new experiments
in the same session — so any NLL change in 10e is attributable to a
single cause. Encouragement here is for *lots* of experimentation,
disciplined.

The notebook has four parts:

1. **Part A — Build the augmented parquet** (Tasks 60–63). Add the new
   conditioning columns (cycle/hemisphere id, opposite-hemisphere
   summaries, smoothed-area trajectory) and write
   `diffusion_windows_v2.parquet` next to this notebook. **Rebuilt every
   run**, never cached, so any change to your augmentation logic
   propagates immediately.
2. **Part B — wandb setup + experiment menu** (Tasks 64–65). Each
   student gets their own wandb project; the experiment menu lists eight
   variants you'll progress through.
3. **Part C — Disciplined sweep** (Task 66). Enable a subset of
   experiments, train each, save a checkpoint per variant.
4. **Part D — Visual sanity check** of the most recent training.

> The **test split is reserved for the PI**. Every cell in this
> notebook filters to `split in {"train", "val"}`. Do not change that.


In [ ]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb


In [ ]:
# ── locate Week 08/09/10 artifacts (same discovery pattern as 10a) ────────
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_10"), ("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if (("week_08" in _base or "week_09" in _base or "week_10" in _base)
            and os.path.isdir(_base) and _base not in _search_dirs):
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path_v1   = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows.parquet",       _parquet_path_v1),
    ("butterflAI_model.py",             _classical_py),
    ("official_model.npz",              _classical_weights),
] if p is None]
if _missing:
    raise FileNotFoundError(f"Cannot locate {_missing}. Searched: {_search_dirs}")

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Make sure we get the *week-10* conditioned_infrastructure (with the
# Extended* classes), not the week-09 template.
if "conditioned_infrastructure" in sys.modules:
    _existing = sys.modules["conditioned_infrastructure"]
    if getattr(_existing, "__file__", None) != _conditioned_py:
        del sys.modules["conditioned_infrastructure"]

from unconditioned_infrastructure import make_cosine_schedule, SampleQualityCallback
from conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    ExtendedConditionalDiffusionLightning,
    sample_conditional_extended,
    build_model,
)
print(f"using conditioned_infrastructure from: {_conditioned_py}")

# Where the v2 parquet will live. By convention this notebook sits in
# weeks/week_10/, and so does the v2 artifact it produces.
_WEEK10_DIR  = os.path.dirname(_conditioned_py)
PARQUET_V2   = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR     = _WEEK10_DIR

from butterflAI_model import ButterflAIModel
classical = ButterflAIModel(_classical_weights)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

# Load the v1 parquet — we extend it but never modify it.
windows_v1 = pd.read_parquet(_parquet_path_v1)
# Test set is reserved for the PI. Filter here, once, and keep that
# filter through the rest of the notebook.
windows_v1 = windows_v1.loc[windows_v1["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v1 parquet (train+val only): {len(windows_v1)} rows")
print(f"  splits: {windows_v1['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v1['cycle'].unique())}")
print(f"v2 parquet target: {PARQUET_V2}")


---
## Part A — Build the augmented parquet

We're going to add three families of new conditioning columns to the
v1 parquet:

1. **Cycle and hemisphere identifiers** — a normalized cycle number and
   a north/south indicator.
2. **Opposite-hemisphere summaries** — for each window, the
   *contemporaneous* opposite-hemisphere activity. This is not leakage:
   contemporaneous opposite-hemisphere activity is operationally
   observable (an operational forecaster on the day of the same window
   would have it).
3. **Smoothed-area trajectory** — the past `K` windows of `area_smoothed`
   for the same hemicycle, as a short autoregressive history.

The build cells **rewrite `diffusion_windows_v2.parquet` every run**. We
deliberately do *not* gate on file existence — if you change how a
column is computed and don't see the change downstream, the most
common explanation is "the file was cached." We avoid that failure
mode by always rewriting.


---
## Task 60 — Cycle and hemisphere identifiers

Add two new columns to the dataframe:

- `cycle_norm`: cycle number rescaled to roughly `[-1, +1]` over the
  train range. The point of normalization is not to be exactly in
  `[-1, +1]` — it's to put the input on the same numerical scale as the
  other conditioning vectors so the network doesn't have to learn an
  outsized weight for it.
- `hemi_id`: `+1` for north, `-1` for south.

Both are cheap and let downstream experiments test whether
*structural* per-cycle / per-hemisphere effects survive once amplitude
is controlled for.


In [ ]:
# Task 60 — add cycle_norm and hemi_id.

windows_aug = windows_v1.copy()

# TODO: compute cycle_norm. Use train-set cycle range so this is well
# defined for both splits. Aim for the train cycle range to map roughly
# onto [-1, +1].
_train_cycles = windows_aug.loc[windows_aug["split"] == "train", "cycle"]
_cmin, _cmax  = _train_cycles.min(), _train_cycles.max()
windows_aug["cycle_norm"] = 2.0 * (windows_aug["cycle"] - _cmin) / (_cmax - _cmin) - 1.0

# TODO: compute hemi_id. +1 north, -1 south.
windows_aug["hemi_id"] = np.where(windows_aug["hemisphere"] == "north", 1.0, -1.0)

print("cycle_norm range:", windows_aug["cycle_norm"].min(), windows_aug["cycle_norm"].max())
print("hemi_id values  :", windows_aug["hemi_id"].unique())


---
## Task 61 — Opposite-hemisphere conditioning

For each window in hemisphere *h* at time `tau_center`, attach the
contemporaneous opposite-hemisphere activity summary:

- `opp_area_smoothed` — opposite hemisphere's `area_smoothed`
- `opp_mu_universal`  — opposite hemisphere's `mu_universal`
- `opp_amplitude`     — opposite hemisphere's `amplitude`
- `opp_valid`         — 1 if a matching opposite row was found at the
  same `(cycle, tau_center)`, else 0.

When `opp_valid == 0` (no matching opposite row), impute with the
**train-set mean** of each opposite-* column. This way the network always
sees a defined input; downstream you can decide whether to gate on the
mask.

**Implementation hint:** the cleanest way is a self-merge of the
dataframe with itself: produce a "right side" with `hemisphere`
flipped and renamed columns, then merge on `(cycle, tau_center)`.


In [ ]:
# Task 61 — attach contemporaneous opposite-hemisphere summaries.

# TODO: build a "right side" dataframe with hemisphere flipped and the
#       three columns we want renamed with the "opp_" prefix.
_flip = {"north": "south", "south": "north"}
_right = (windows_aug
          .loc[:, ["cycle", "tau_center", "hemisphere",
                   "area_smoothed", "mu_universal", "amplitude"]]
          .assign(opp_of_hemisphere=lambda d: d["hemisphere"].map(_flip))
          .drop(columns=["hemisphere"])
          .rename(columns={
              "area_smoothed": "opp_area_smoothed",
              "mu_universal":  "opp_mu_universal",
              "amplitude":     "opp_amplitude",
              "opp_of_hemisphere": "hemisphere",
          }))

# TODO: merge on (cycle, tau_center, hemisphere) so each row gets the
#       opposite-side row that has the *flipped* hemisphere stored under
#       the same hemisphere key.
windows_aug = windows_aug.merge(
    _right, on=["cycle", "tau_center", "hemisphere"],
    how="left", indicator="_opp_match",
)
windows_aug["opp_valid"] = (windows_aug["_opp_match"] == "both").astype(np.float32)
windows_aug = windows_aug.drop(columns=["_opp_match"])

# TODO: impute missing opp_* values with train-set means.
_opp_cols = ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"]
_train_mask = (windows_aug["split"] == "train") & (windows_aug["opp_valid"] == 1.0)
for _c in _opp_cols:
    _mean = windows_aug.loc[_train_mask, _c].mean()
    windows_aug[_c] = windows_aug[_c].fillna(_mean)

print(f"opp_valid coverage: {windows_aug['opp_valid'].mean():.3f}")
print(f"opp_area_smoothed (train, valid): "
      f"mean={windows_aug.loc[_train_mask, 'opp_area_smoothed'].mean():.3f}, "
      f"std={windows_aug.loc[_train_mask, 'opp_area_smoothed'].std():.3f}")


---
## Task 62 — Smoothed-area trajectory (cycle history)

For each window, attach the previous `K` values of `area_smoothed`
from the same hemicycle (same `cycle` AND same `hemisphere`), ordered
chronologically by `tau_center`. Columns: `area_lag1`, `area_lag2`, …,
`area_lagK`.

`area_lag1` is the *previous* window's smoothed area; `area_lagK` is
the one K steps back. Boundary windows (near the start of a hemicycle,
where fewer than K prior windows exist) get train-set-mean imputation
and a `traj_valid` column set to 0 for that row.

The point: cycle *history* is information that amplitude alone misses.
A window 6 months into a strong cycle and a window 6 months from the
end of a strong cycle have similar amplitude but very different
trajectory.

**Implementation hint:** sort within each hemicycle by `tau_center`,
then shift the `area_smoothed` series by 1, 2, …, K.


In [ ]:
# Task 62 — attach the smoothed-area trajectory (K lagged values).

K_LAGS = 4

# TODO: within each (cycle, hemisphere) group, sort by tau_center and
#       produce K columns of shifted area_smoothed.
_sorted = windows_aug.sort_values(["cycle", "hemisphere", "tau_center"])

_lag_cols = [f"area_lag{k}" for k in range(1, K_LAGS + 1)]
for k, col in enumerate(_lag_cols, start=1):
    _sorted[col] = _sorted.groupby(["cycle", "hemisphere"])["area_smoothed"].shift(k)

# A row is traj_valid iff *all* K lags exist.
_sorted["traj_valid"] = (~_sorted[_lag_cols].isna().any(axis=1)).astype(np.float32)

# TODO: impute boundary NaNs with train-set means.
_train_mask = (_sorted["split"] == "train") & (_sorted["traj_valid"] == 1.0)
for col in _lag_cols:
    _mean = _sorted.loc[_train_mask, col].mean()
    _sorted[col] = _sorted[col].fillna(_mean)

windows_aug = _sorted.sort_index()  # restore original row order

print(f"traj_valid coverage: {windows_aug['traj_valid'].mean():.3f}")
print(f"lag columns: {_lag_cols}")


---
## Task 63 — Write `diffusion_windows_v2.parquet` and sanity-check

Write the augmented dataframe. Sanity checks before we trust it
downstream:

- Row count unchanged from v1 (we did not gain or lose any windows).
- Every original v1 column is preserved bit-for-bit.
- New cond columns are finite **wherever the validity mask says they
  should be**.
- `split` column is unchanged.


In [ ]:
# Task 63 — write v2 + sanity checks.

# Sanity 1: row count.
assert len(windows_aug) == len(windows_v1), \
    f"row count drifted: {len(windows_aug)} vs v1 {len(windows_v1)}"

# Sanity 2: every v1 column preserved bit-for-bit.
for c in windows_v1.columns:
    assert c in windows_aug.columns, f"missing v1 column {c}"
    if windows_v1[c].dtype.kind in "fc":
        assert np.allclose(windows_v1[c].to_numpy(),
                           windows_aug[c].to_numpy(), equal_nan=True), c
    else:
        assert (windows_v1[c].astype(str).to_numpy()
                == windows_aug[c].astype(str).to_numpy()).all(), c

# Sanity 3: new columns finite where the validity masks allow.
NEW_COND_COLS = ["cycle_norm", "hemi_id",
                 "opp_area_smoothed", "opp_mu_universal", "opp_amplitude",
                 *[f"area_lag{k}" for k in range(1, K_LAGS + 1)]]
for c in NEW_COND_COLS:
    assert windows_aug[c].notna().all(), f"NaN remains in {c}"

# Sanity 4: split column unchanged.
assert (windows_aug["split"].to_numpy() == windows_v1["split"].to_numpy()).all()

# Always rewrite. Never cache.
windows_aug.to_parquet(PARQUET_V2, index=False)
print(f"wrote {PARQUET_V2}  ({len(windows_aug)} rows, {len(windows_aug.columns)} cols)")
print(f"new cond columns: {NEW_COND_COLS}")


---
## Part B — wandb setup and the experiment menu

### Task 64 — Per-student wandb project

Each student gets their **own** wandb project. Replace the placeholder
strings below with your handle. Every training run in this notebook
logs to that project with the experiment ID as the run name; you can
compare all your variants on a single dashboard.

If wandb is unavailable in your environment, training will fall back to
a local CSV logger automatically.


In [ ]:
# Task 64 — wandb identity. EDIT THESE.

WANDB_PROJECT = "butterflai-w10ext-<your-handle>"
WANDB_ENTITY  = "<your-wandb-username>"      # set to None if you don't use teams

assert "your-handle" not in WANDB_PROJECT, \
    "Set WANDB_PROJECT to your own project name before training."


### Task 65 — The experiment menu

The eight experiments below are defined as config dicts. They are
ordered from *cheapest to add* to *biggest combined bet*. Each one
differs from the Week 10 baseline by **a single knob** — that is the
whole point of doing them one at a time.

| # | Name | Change | Why |
|---|------|--------|-----|
| **E0** | Baseline reproduce | Same 4-D cond on v2 parquet | Sanity check — v2 + new infrastructure should not regress NLL |
| **E1** | + cycle/hemi id | Adds `cond_cyclehemi` | Per-cycle / hemispheric effects beyond amplitude |
| **E2** | + opposite hemisphere | Adds `cond_opp` | Same-time opposite hemisphere informs same-side residuals |
| **E3** | + smoothed-area trajectory | Adds `cond_traj` | Cycle *history*; amplitude alone misses time dynamics |
| **E4** | FiLM (same 4-D cond) | Same cond as Week 10, FiLM modulation | Architecture vs information — does the mechanism alone close the gap? |
| **E5** | FiLM + best cond | FiLM + whichever of E1–E3 won on val | Combined best; only meaningful after E1–E4 ran |
| **E6** | Classifier-free guidance | Train w/ `cond_dropout_p=0.1`; 10e sweeps `guidance_w` | Sharpens conditional density |
| **E7** | + Fourier features | sin/cos on cond scalars before the network | Often helps continuous-scalar conditioning |

Read the configs; understand which dataset groups each one needs and
which architecture it uses. Don't enable them yet.


In [ ]:
# Task 65 — experiment menu (do NOT edit unless you're proposing a new variant).

# A single shared template; each experiment overrides only what it changes.
_BASE_TEMPLATE = {
    "arch":          "concat",
    "consumed_keys": ["cond_base"],
    "groups":        ["base"],
    "hidden_dim":    128,
    "n_layers":      3,
    "fourier":       False,
    "cond_dropout_p": 0.0,
    "max_epochs":    20000,
    "lr":            1e-3,
    "batch_size":    64,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    "E0": _spec(),                                                       # Week-10 baseline reproduced on v2
    "E1": _spec(consumed_keys=["cond_base", "cond_cyclehemi"],
                groups=["base", "cyclehemi"]),
    "E2": _spec(consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"]),
    "E3": _spec(consumed_keys=["cond_base", "cond_traj"],
                groups=["base", "traj"]),
    "E4": _spec(arch="film"),                                            # same 4-D cond, FiLM modulation
    # E5 — set consumed_keys/groups to your winning E1/E2/E3 + film
    "E5": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"]),
    "E6": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"],
                cond_dropout_p=0.1),
    "E7": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"],
                fourier=True),
}

for name, cfg in EXPERIMENTS.items():
    print(f"{name}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")


---
## Part C — Disciplined sweep

### Task 66 — Enable a subset and train

The discipline:

1. **First run ever.** `ENABLED_EXPERIMENTS = ["E0"]`. Verify in 10e
   that E0's NLL is within noise of Week 10's baseline. If it isn't,
   stop and figure out why before adding more experiments — you'd be
   chasing changes in two places at once.
2. **Next session.** Add **one** of E1, E2, E3, or E4. Re-run. Compare
   in 10e.
3. **Repeat.** Add at most one new experiment per session. Different
   students should pick different ones — results pool in 10e.
4. **After E1–E3 each have a val NLL.** Update E5's `consumed_keys` /
   `groups` to your winning cond set, then enable E5.
5. **Last.** E6 (CFG) and E7 (Fourier).

The loop skips checkpoints that already exist on disk — re-running
this notebook does *not* retrain unless you delete the file.


In [ ]:
# Task 66 — enable, then train. EDIT THIS LIST.

ENABLED_EXPERIMENTS = ["E0"]   # start here, then add one at a time

from infrastructure.utils.reproducibility import set_all_seeds


class CondSampleQualityCallback(SampleQualityCallback):
    """Same shape as the Week 10 callback but samples via the
    CFG-capable extended sampler. Each callback instance owns a fixed
    batch of (already-concatenated, train-set-normalized) cond vectors
    so the comparison is apples-to-apples across epochs."""
    def __init__(self, train_samples, cond_reference, **kw):
        super().__init__(train_samples, **kw)
        self._cond_ref = cond_reference.detach().clone()
    def _sample(self, pl_module):
        device = next(pl_module.parameters()).device
        return sample_conditional_extended(pl_module, self._cond_ref,
                                            guidance_w=0.0, device=device).cpu().numpy()


def _train_one(name, cfg):
    ckpt_path = os.path.join(CKPT_DIR, f"ckpt_{name}.ckpt")
    if os.path.isfile(ckpt_path):
        print(f"[{name}] checkpoint exists at {ckpt_path}; skipping training.")
        return ckpt_path

    set_all_seeds(42)

    # Datasets in dict-of-groups form.
    train_ds = ExtendedConditionalResidualDataset(windows_aug, "train", groups=cfg["groups"])
    val_ds   = ExtendedConditionalResidualDataset(windows_aug, "val",   groups=cfg["groups"],
                                                  group_stats=train_ds.group_stats)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=cfg["batch_size"],
                                               shuffle=True,  num_workers=0)
    val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=cfg["batch_size"],
                                               shuffle=False, num_workers=0)

    total_dim = sum(train_ds.group_dims[k.replace("cond_", "")] for k in cfg["consumed_keys"])

    model = build_model(cfg, total_cond_dim=total_dim)
    lit = ExtendedConditionalDiffusionLightning(
        model=model, alpha=alpha_np, sigma=sigma_np, T=T,
        consumed_keys=cfg["consumed_keys"],
        group_stats={g: train_ds.group_stats[g] for g in cfg["groups"]
                     if f"cond_{g}" in cfg["consumed_keys"]},
        total_cond_dim=total_dim,
        bin_means=train_ds.bin_means, bin_stds=train_ds.bin_stds,
        lr=cfg["lr"], scheduler="cosine", weight_decay=1e-4,
        cond_dropout_p=cfg["cond_dropout_p"],
    )

    # Sample-quality callback (uses train cond as reference batch).
    bin_means_np = train_ds.bin_means.numpy(); bin_stds_np = train_ds.bin_stds.numpy()
    all_tr_std  = torch.stack([train_ds[i]["r_clean"] for i in range(len(train_ds))]).numpy()
    all_tr_phys = all_tr_std * bin_stds_np + bin_means_np
    all_va_std  = torch.stack([val_ds[i]["r_clean"]   for i in range(len(val_ds))]).numpy()
    all_va_phys = all_va_std * bin_stds_np + bin_means_np
    cond_ref    = torch.cat([torch.stack([train_ds[i][k] for i in range(min(500, len(train_ds)))])
                             for k in cfg["consumed_keys"]], dim=-1)

    cb = CondSampleQualityCallback(
        train_samples=all_tr_phys, val_samples=all_va_phys,
        cond_reference=cond_ref,
        every_n_epochs=200, n_compare=cond_ref.shape[0],
        bin_centers=BIN_CENTERS, bin_width=BIN_WIDTH,
    )

    try:
        logger = WandbLogger(
            project=WANDB_PROJECT, entity=WANDB_ENTITY,
            name=name, save_dir=os.path.join(_WEEK10_DIR, "wandb_logs"),
        )
    except Exception as _e:
        print(f"[{name}] WandB unavailable ({_e}); falling back to CSVLogger.")
        logger = CSVLogger(os.path.join(_WEEK10_DIR, "csv_logs"), name=name)

    trainer = pl.Trainer(
        max_epochs=cfg["max_epochs"], logger=logger,
        accelerator="auto", devices="auto",
        log_every_n_steps=10, enable_progress_bar=False,
        callbacks=[cb],
    )
    trainer.fit(lit, train_loader, val_loader)
    trainer.save_checkpoint(ckpt_path)
    try:
        wandb.finish()
    except Exception:
        pass
    print(f"[{name}] saved {ckpt_path}")
    return ckpt_path


for _name in ENABLED_EXPERIMENTS:
    if _name not in EXPERIMENTS:
        raise KeyError(f"unknown experiment {_name!r}; defined: {list(EXPERIMENTS)}")
    _train_one(_name, EXPERIMENTS[_name])


---
## Part D — Visual sanity check on the most recent training

Sample a small batch of validation conditioning vectors and overlay
the diffusion's generated residuals against the ground truth. This is
a "did training collapse?" check — not a quantitative comparison. The
real evaluation lives in 10e.


In [ ]:
# Part D — quick overlay for the most recently trained checkpoint.

if not ENABLED_EXPERIMENTS:
    print("No experiments were trained this session — nothing to visualize.")
else:
    _name = ENABLED_EXPERIMENTS[-1]
    _cfg  = EXPERIMENTS[_name]
    _ckpt = os.path.join(CKPT_DIR, f"ckpt_{_name}.ckpt")

    # Rebuild dataset/model in the same shape we trained, then load weights.
    train_ds = ExtendedConditionalResidualDataset(windows_aug, "train", groups=_cfg["groups"])
    val_ds   = ExtendedConditionalResidualDataset(windows_aug, "val",   groups=_cfg["groups"],
                                                  group_stats=train_ds.group_stats)
    total_dim = sum(train_ds.group_dims[k.replace("cond_", "")] for k in _cfg["consumed_keys"])

    model = build_model(_cfg, total_cond_dim=total_dim)
    lit = ExtendedConditionalDiffusionLightning.load_from_checkpoint(
        _ckpt, model=model, alpha=alpha_np, sigma=sigma_np,
        group_stats={g: train_ds.group_stats[g] for g in _cfg["groups"]
                     if f"cond_{g}" in _cfg["consumed_keys"]},
        total_cond_dim=total_dim, consumed_keys=_cfg["consumed_keys"],
    )

    # Sample on a small batch of val windows.
    n_show = 4
    cond_concat = torch.cat([torch.stack([val_ds[i][k] for i in range(n_show)])
                             for k in _cfg["consumed_keys"]], dim=-1)
    truth = torch.stack([val_ds[i]["r_clean"] for i in range(n_show)]).numpy()
    truth_phys = truth * val_ds.bin_stds.numpy() + val_ds.bin_means.numpy()

    samples = sample_conditional_extended(lit, cond_concat, guidance_w=0.0).cpu().numpy()

    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3.2), sharey=True)
    for i, ax in enumerate(axes):
        w = BIN_WIDTH * 0.4
        ax.bar(BIN_CENTERS - w / 2, truth_phys[i],  width=w, color="C0", label="truth")
        ax.bar(BIN_CENTERS + w / 2, samples[i],     width=w, color="C2", label="sampled")
        ax.axhline(0, color="k", lw=0.4)
        ax.set_title(f"val window {i}")
        ax.set_xlabel("|latitude| (°)")
    axes[0].set_ylabel("residual")
    axes[0].legend()
    fig.suptitle(f"{_name}: visual sanity check")
    fig.tight_layout(); plt.show()


---
## Handoff to `10e_diffusion_NLL_ablations.ipynb`

Each enabled experiment has produced a checkpoint named
`ckpt_E{N}.ckpt` in this directory. Notebook 10e will:

- Train a small **oracle MLP** per experiment (cond → Gaussian residual
  params): the resulting NLL is an *upper bound* on what any diffusion
  with that cond set could achieve.
- Score every `ckpt_E*.ckpt` on the val split (K=100 conditional
  samples per window) and produce the headline NLL bar chart with the
  classical baseline and Week 10's existing checkpoint as anchors.
- Run the guidance sweep for any E6 checkpoint.

**The test split is reserved for the PI.** Do not modify the
`split.isin(["train", "val"])` filter at the top of this notebook, and
do not introduce one in 10e.
